<a href="https://colab.research.google.com/github/melisms/techIstanbulBootcamp/blob/main/Battal_Hoca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Derin Öğrenme ve Büyük Dil Modelleri
### Canlı eğitim · Uygulama defteri

Bu defter sunumdaki bölümlerin **uygulamasıdır**. Her bölümde önce sunumda konuyu anlatırız, sonra bu defterde aynı numaralı bölüme gelip hücreleri **yukarıdan aşağıya sırayla** çalıştırırız.

**Defteri nasıl okumalı?**
- Her kod satırının yanında `#` ile başlayan bir açıklama var: o satırın **ne yaptığı** ve **neden** yazıldığı.
- Bir fonksiyona birden fazla bilgi (parametre) veriyorsak her parametreyi ayrı satıra yazdık ve yanına ne işe yaradığını ekledik.
- Bir hücreyi çalıştırmak için hücreye tıklayıp `Shift + Enter` tuşlarına basın.

| Saat | Bölüm | Konular |
|:-:|---|---|
| 00:00 | Açılış | Ders planı |
| 00:10 | 1. Derin öğrenmenin temelleri | Deep learning mantığı, yapay sinir ağları, nöron, katman, ağırlık, bias, aktivasyon fonksiyonları |
| 00:45 | 2. Bir ağ nasıl öğrenir? | İleri yayılım, kayıp fonksiyonları, geri yayılım |
| 01:20 | *Mola* | |
| 01:30 | 3. Keras ile ilk model | TensorFlow/Keras ve PyTorch |
| 01:55 | 4. NLP'ye giriş | Tokenization, embedding |
| 02:20 | *Mola* | |
| 02:30 | 5. Transformer | Attention, Transformer mimarisi, context window |
| 03:00 | 6. Büyük dil modelleri | LLM çalışma mantığı, GPT tipi modeller |
| 03:30 | **7. Proje** | **Müşteri mesajlarını otomatik yönlendiren yapay zekâ** |
| 03:50 | Kapanış | Özet, soru-cevap |

> **Dersten önce:** `Çalışma zamanı → Tümünü çalıştır` ile defteri bir kez baştan sona çalıştırın. Veri seti ve hazır modeller (toplam ~2.7 GB) önceden iner, derste beklemezsiniz. GPU gerekmez; model indirirken çıkabilecek `HF_TOKEN` uyarısı önemsizdir.

## 1. Derin Öğrenmenin Temelleri

**Deep learning (derin öğrenme)**, çok katmanlı yapay sinir ağlarıyla veriden öğrenen bir makine öğrenmesi yöntemidir. Klasik programlamada kuralları biz yazarız; derin öğrenmede ise model, örneklere bakarak kuralları **kendisi** bulur.

Bu bölümde bir yapay sinir ağının en küçük parçası olan **nöronu** elle hesaplayacağız. Ardından nöronların bir araya gelmesiyle oluşan **katmanı** ve nöronun çıkışını belirleyen **aktivasyon fonksiyonlarını** göreceğiz.

### 1.1 Hazırlık: kütüphaneleri yüklemek

**Bu hücrede:** Ders boyunca kullanacağımız iki temel kütüphaneyi yüklüyoruz.

In [ ]:
import numpy as np               # NumPy: sayı dizileri (vektör, matris) üzerinde hızlı hesap yapar.
                                 # "as np" → her seferinde "numpy" yerine kısaca "np" yazmak için.
import matplotlib.pyplot as plt  # Matplotlib: grafik çizer. "as plt" → kısaca "plt" diye kullanmak için.

np.random.seed(42)               # Rastgele sayı üretecini sabitler; böylece herkes aynı sonucu görür.
print("Kütüphaneler hazır.")     # Hücrenin sorunsuz çalıştığını görmek için kısa bir mesaj yazdırır.

### 1.2 Tek bir yapay nöron

Nöron iki adımda çalışır:
1. **Ağırlıklı toplam:** Her girdiyi kendi ağırlığıyla çarpar, sonuçları toplar ve bias ekler: `z = w₁·x₁ + w₂·x₂ + b`
2. **Aktivasyon:** Toplamı bir fonksiyondan geçirir: `a = sigmoid(z)`

**Örnek:** Bir öğrencinin haftalık çalışma saati ve günlük uyku saatinden sınavı geçme olasılığını tahmin eden bir nöron. Ağırlıkları şimdilik biz veriyoruz; gerçekte bunları eğitim bulur (Bölüm 2).

In [ ]:
x = np.array([10, 7])      # Girdiler (x): 10 saat çalışma, 7 saat uyku.
                           # np.array → sayı listesini NumPy vektörüne çevirir.
w = np.array([0.4, 0.3])   # Ağırlıklar (w): her girdinin önemi. Çalışma saati (0.4) biraz daha etkili.
b = -5                     # Bias (b): nöronun eşik ayarı; toplamı yukarı ya da aşağı kaydırır.

z = np.dot(w, x) + b       # 1. adım – ağırlıklı toplam. np.dot(w, x) = 0.4·10 + 0.3·7 = 6.1
                           # Sonra bias eklenir: 6.1 + (−5) = 1.1

a = 1 / (1 + np.exp(-z))   # 2. adım – sigmoid aktivasyonu: sonucu 0 ile 1 arasına sıkıştırır.
                           # np.exp(-z) → e sayısının −z'inci kuvveti. Çıktıyı olasılık gibi okuruz.

print("Ağırlıklı toplam z :", round(z, 2))   # round(sayı, 2) → sayıyı 2 ondalık basamağa yuvarlar.
print("Nöronun çıktısı a  :", round(a, 2))   # 0.75 → sınavı geçme olasılığı yaklaşık %75.

**Sonuç:** Nöron `z = 1.1` toplamını sigmoid ile `0.75`'e çevirdi. Ağırlıkları veya bias'ı değiştirip hücreyi yeniden çalıştırın: sonuç nasıl değişiyor?

### 1.3 Katman: aynı girdiyi işleyen birçok nöron

Bir **katman**, aynı girdileri alan birden fazla nörondan oluşur. Her nöronun kendi ağırlıkları ve bias'ı vardır. Hepsini tek tek hesaplamak yerine **matris çarpımıyla** tek satırda hesaplarız.

**Bu hücrede:** Aynı iki girdiyi 3 nöronlu bir katmandan geçiriyoruz.

In [ ]:
x = np.array([10, 7])              # Aynı girdiler: çalışma ve uyku saati.

W = np.array([[0.4, -0.2,  0.1],   # Ağırlık matrisi: 2 satır (girdi sayısı) × 3 sütun (nöron sayısı).
              [0.3,  0.5, -0.1]])  # Her SÜTUN bir nöronun ağırlıklarıdır (1. nöron: 0.4 ve 0.3).
b = np.array([-5, -1, 0])          # Her nöronun kendi bias'ı: 3 nöron → 3 bias.

z = x @ W + b                      # "@" matris çarpımıdır: 3 nöronun toplamını tek seferde hesaplar.
a = 1 / (1 + np.exp(-z))           # Sigmoid, her nöronun toplamına ayrı ayrı uygulanır.

print("Ağırlık matrisinin boyutu:", W.shape)   # .shape → boyut: (2, 3) = 2 girdi, 3 nöron.
print("Katmanın toplamları z    :", np.round(z, 2))   # np.round → dizideki her sayıyı yuvarlar.
print("Katmanın çıktıları a     :", np.round(a, 2))   # 3 nöron → 3 çıktı.

**Sonuç:** Tek satırlık `x @ W + b` işlemi 3 nöronun hesabını birlikte yaptı. Nöron sayısı arttıkça yalnızca `W` matrisi büyür. Ağdaki tüm ağırlık ve bias sayılarına **parametre** denir; eğitim sırasında öğrenilen şey bu sayılardır.

### 1.4 Aktivasyon fonksiyonları

Aktivasyon fonksiyonu nöronun toplamını (`z`) çıktıya (`a`) çevirir ve ağa **doğrusal olmayan** bir davranış kazandırır. Aktivasyon olmasaydı ağ ne kadar derin olursa olsun yalnızca düz çizgiler çizebilirdi.

**Bu hücrede:** En çok kullanılan üç aktivasyon fonksiyonunu tanımlayıp aynı grafikte çiziyoruz.

In [ ]:
def sigmoid(z):                    # def → kendi fonksiyonumuzu tanımlar; z fonksiyonun girdisidir.
    return 1 / (1 + np.exp(-z))    # Sonucu 0 ile 1 arasına sıkıştırır (olasılık gibi).

def relu(z):
    return np.maximum(0, z)        # Negatif değerleri 0 yapar, pozitifleri olduğu gibi bırakır.

def tanh(z):
    return np.tanh(z)              # Sonucu −1 ile 1 arasına sıkıştırır.

z = np.linspace(-5, 5, 100)        # −5 ile 5 arasında eşit aralıklı 100 sayı üretir (yatay eksen).

plt.plot(                          # plot → çizgi grafiği çizer.
    z,                             # Yatay eksendeki değerler.
    sigmoid(z),                    # Dikey eksendeki değerler: her z için sigmoid sonucu.
    label="Sigmoid",               # label → çizginin lejanttaki (açıklama kutusu) adı.
)
plt.plot(z, relu(z), label="ReLU")         # Aynı şekilde ReLU çizgisi (x, y, ad).
plt.plot(z, tanh(z), label="Tanh")         # Aynı şekilde tanh çizgisi (x, y, ad).
plt.title("Aktivasyon fonksiyonları")      # Grafiğin başlığı.
plt.xlabel("z (nöronun toplamı)")          # Yatay eksenin adı.
plt.ylabel("a (nöronun çıktısı)")          # Dikey eksenin adı.
plt.ylim(-1.5, 3)                          # Dikey ekseni −1.5 ile 3 arasında göster (ReLU çok yükselmesin).
plt.grid(True)                             # Arka plana ızgara çizgileri ekler; değer okumak kolaylaşır.
plt.legend()                               # Lejantı (hangi renk hangi fonksiyon) gösterir.
plt.show()                                 # Grafiği ekrana çizer.

**Sonuç:**
- **Sigmoid** çıktıyı 0–1 arasına sıkıştırır; evet/hayır sorularının (ikili sınıflandırma) çıkış katmanında kullanılır.
- **ReLU** negatifleri sıfırlar; basit ve hızlı olduğu için gizli katmanlarda en yaygın tercihtir.
- **Tanh** çıktıyı −1 ile 1 arasına sıkıştırır.

## 2. Bir Ağ Nasıl Öğrenir?

Öğrenme dört adımlı bir döngüdür:
1. **İleri yayılım:** Girdiden bir tahmin üret.
2. **Kayıp:** Tahminin ne kadar yanlış olduğunu ölç.
3. **Geri yayılım:** Her ağırlığın bu hataya ne kadar katkı yaptığını bul.
4. **Güncelleme:** Ağırlıkları hatayı azaltacak yönde biraz değiştir.

### 2.1 İleri yayılım (forward propagation)

Girdinin katmanlardan sırayla geçerek tahmine dönüşmesidir. Her katmanda aynı iki iş yapılır: ağırlıklı toplam, ardından aktivasyon.

**Bu hücrede:** 2 girdi → 3 nöronlu gizli katman (ReLU) → 1 nöronlu çıkış katmanı (sigmoid) yapısındaki küçük bir ağın tahminini adım adım hesaplıyoruz.

In [ ]:
x = np.array([1.0, 2.0])                 # Girdi: 2 özellik.

W1 = np.array([[0.2, -0.4,  0.7],        # Gizli katmanın ağırlıkları: 2 girdi × 3 nöron.
               [0.5, -0.3, -0.2]])
b1 = np.array([0.1, 0.0, -0.1])          # Gizli katmanın bias'ları: 3 nöron → 3 bias.

W2 = np.array([[0.6], [-0.3], [0.8]])    # Çıkış katmanının ağırlıkları: 3 girdi × 1 nöron.
b2 = np.array([0.05])                    # Çıkış nöronunun bias'ı.

z1 = x @ W1 + b1          # Adım 1: gizli katmanın ağırlıklı toplamları.
a1 = relu(z1)             # Adım 2: ReLU → negatif toplam 0 olur (o nöron "susar").
z2 = a1 @ W2 + b2         # Adım 3: çıkış nöronunun ağırlıklı toplamı.
y_tahmin = sigmoid(z2)    # Adım 4: sigmoid → 0 ile 1 arası bir tahmin.

print("Gizli katman toplamları z1:", np.round(z1, 2))
print("ReLU sonrası a1           :", np.round(a1, 2))
print("Ağın tahmini              :", np.round(y_tahmin, 2))

**Sonuç:** İkinci nöronun toplamı −1.0 olduğu için ReLU onu 0 yaptı. Ağın tahmini **0.73**. Keras'taki `model.predict()` da tam olarak bu hesabı yapar (Bölüm 3).

### 2.2 Kayıp fonksiyonları (loss)

Kayıp, tahminin gerçek değerden **ne kadar uzak** olduğunu tek bir sayıyla ölçer. Eğitimin amacı bu sayıyı küçültmektir.
- **MSE (ortalama kare hata):** Sayı tahmini (regresyon) problemlerinde kullanılır.
- **Çapraz entropi (cross-entropy):** Sınıflandırma problemlerinde kullanılır.

**Bu hücrede:** İki kayıp türünü küçük örneklerle elle hesaplıyoruz.

In [ ]:
# --- MSE: sayı tahmininde ---
gercek = np.array([3.0, 5.0, 7.0])    # Gerçek değerler (ör. ev fiyatı, milyon TL).
tahmin = np.array([2.5, 5.0, 8.0])    # Modelin tahminleri.
hata = gercek - tahmin                # Her tahmindeki hata: [0.5, 0, −1].
mse = np.mean(hata ** 2)              # Hataların karesini al (** 2), sonra ortalamasını bul (np.mean).
print("MSE:", round(mse, 2))          # Kare alındığı için büyük hatalar daha ağır cezalandırılır.

# --- Çapraz entropi: sınıflandırmada ---
# Bir kedi resmi için model 3 sınıfa olasılık veriyor: [kedi, köpek, kuş]
iyi_tahmin = np.array([0.8, 0.15, 0.05])   # Doğru sınıfa (kedi) yüksek olasılık vermiş.
kotu_tahmin = np.array([0.1, 0.7, 0.2])    # Doğru sınıfa düşük olasılık vermiş.

# Çapraz entropi = −log(doğru sınıfa verilen olasılık). Doğru sınıf "kedi" → listenin 0. elemanı.
print("İyi tahminin kaybı :", round(-np.log(iyi_tahmin[0]), 2))    # np.log → doğal logaritma.
print("Kötü tahminin kaybı:", round(-np.log(kotu_tahmin[0]), 2))   # Olasılık küçüldükçe kayıp büyür.

**Sonuç:** Doğru cevaba %80 olasılık veren modelin kaybı küçük (0.22), %10 veren modelin kaybı büyük (2.30). Büyük dil modelleri de bu kayıpla eğitilir: "doğru sonraki kelimeye ne kadar olasılık verdin?" (Bölüm 6).

### 2.3 Geri yayılım ve gradyan inişi: ağın öğrenmesi

- **Geri yayılım (backpropagation):** Kaybın her ağırlığa göre türevini, yani **gradyanını** hesaplar. Gradyan şu sorunun cevabıdır: "Bu ağırlığı biraz artırırsam kayıp artar mı, azalır mı, ne kadar?"
- **Gradyan inişi:** Her ağırlığı gradyanın tersi yönünde küçük bir adımla günceller: `w = w − öğrenme_oranı × gradyan`

**Bu hücrede:** Tek nöronlu bir modele `y = 2x + 1` ilişkisini öğretiyoruz. Model `w` ve `b` değerlerini bilmiyor, 0'dan başlıyor ve yalnızca örneklere bakarak öğreniyor.

In [ ]:
# Eğitim verisi: y = 2x + 1 kuralıyla üretilmiş 5 örnek (kuralı modele söylemiyoruz!)
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])    # Girdiler.
y = np.array([3.0, 5.0, 7.0, 9.0, 11.0])   # Doğru cevaplar: her biri 2·x + 1.

w = 0.0                  # Ağırlık: bilinmiyor, 0'dan başlıyoruz.
b = 0.0                  # Bias: bilinmiyor, 0'dan başlıyoruz.
ogrenme_orani = 0.05     # Her güncellemede atılacak adımın büyüklüğü.
kayiplar = []            # Her adımdaki kaybı saklamak için boş liste (sonra grafiğini çizeceğiz).

for adim in range(300):                     # Aynı dört adımı 300 kez tekrarla.
    y_tahmin = w * x + b                    # 1) İleri yayılım: tahmin üret.
    kayip = np.mean((y_tahmin - y) ** 2)    # 2) Kayıp: MSE ile hatayı ölç.
    kayiplar.append(kayip)                  #    .append → kaybı listenin sonuna ekler.

    # 3) Geri yayılım: MSE'nin w'ye ve b'ye göre türevi (gradyan)
    gradyan_w = np.mean(2 * (y_tahmin - y) * x)   # Kaybın w'ye göre türevi.
    gradyan_b = np.mean(2 * (y_tahmin - y))       # Kaybın b'ye göre türevi.

    # 4) Güncelleme (gradyan inişi): gradyanın tersi yönünde küçük bir adım
    w = w - ogrenme_orani * gradyan_w   # Gradyan pozitifse w azalır, negatifse artar.
    b = b - ogrenme_orani * gradyan_b   # b için de aynı kural.

    if adim % 50 == 0:                      # % → bölümden kalan. Her 50 adımda bir durumu yazdır.
        print("Adım", adim, "| kayıp:", round(kayip, 3), "| w:", round(w, 2), "| b:", round(b, 2))

print("Öğrenilen w:", round(w, 2), "| Öğrenilen b:", round(b, 2))   # Hedef: w = 2, b = 1

**Bu hücrede:** Eğitim boyunca kaybın nasıl azaldığını çiziyoruz.

In [ ]:
plt.plot(kayiplar)                  # Kayıp listesini çizer; yatay eksen otomatik olarak adım numarası olur.
plt.title("Eğitim boyunca kayıp")   # Grafiğin başlığı.
plt.xlabel("Adım")                  # Yatay eksen: eğitim adımı.
plt.ylabel("Kayıp (MSE)")           # Dikey eksen: o adımdaki hata.
plt.grid(True)                      # Izgara çizgileri.
plt.show()                          # Grafiği ekrana çizer.

**Sonuç:** Model `w`'yi 2'ye, `b`'yi 1'e yaklaştırdı; yani `y = 2x + 1` kuralını örneklerden kendisi buldu. Gerçek derin ağlarda milyonlarca ağırlık vardır ama döngü aynıdır. Türevleri elle yazmak yerine TensorFlow ve PyTorch bunları otomatik hesaplar.

## 3. Keras ile İlk Derin Öğrenme Modeli

**TensorFlow** (Google) ve **PyTorch** (Meta), en yaygın iki derin öğrenme kütüphanesidir. İkisi de ileri yayılımı, geri yayılımı ve güncellemeyi bizim için otomatik yapar. **Keras**, TensorFlow üzerinde çalışan ve model kurmayı çok kısaltan bir arayüzdür.

Bu bölümde Keras ile el yazısı rakamları tanıyan bir model kuracağız. Keras'ta 4 adım vardır: **veriyi hazırla → modeli tanımla → derle (`compile`) → eğit (`fit`)**.

### 3.1 Veriyi hazırlamak: MNIST

**Bu hücrede:** 70.000 el yazısı rakam görüntüsünden oluşan MNIST veri setini indiriyor ve bir örneğe bakıyoruz.

In [ ]:
import keras                               # Keras: TensorFlow üzerinde çalışan model kurma kütüphanesi.

(X_egitim, y_egitim), (X_test, y_test) = keras.datasets.mnist.load_data()
# load_data() → veriyi indirir ve iki parçaya ayırır:
#   X_egitim, y_egitim → modelin öğreneceği 60.000 görüntü ve doğru rakamları (etiketler)
#   X_test, y_test     → modelin hiç görmeyeceği 10.000 görüntü: başarıyı ölçmek için

print("Eğitim görüntüleri:", X_egitim.shape)    # (60000, 28, 28) → 60.000 görüntü, her biri 28×28 piksel.
print("İlk görüntünün etiketi:", y_egitim[0])   # Görüntüdeki rakamın doğru cevabı.

X_egitim = X_egitim / 255.0   # Piksel değerleri 0–255 arasında; 255'e bölerek 0–1 arasına getiriyoruz.
X_test = X_test / 255.0       # Küçük ve benzer ölçekteki sayılarla model daha kolay öğrenir.

plt.imshow(                                 # imshow → sayı matrisini resim olarak çizer.
    X_egitim[0],                            # Gösterilecek görüntü: eğitim setinin ilk görüntüsü.
    cmap="gray",                            # Renk haritası: gri tonları (siyah-beyaz).
)
plt.title("Etiket: " + str(y_egitim[0]))    # str() → sayıyı yazıya çevirir ki başlığa eklenebilsin.
plt.axis("off")                             # Eksen çizgilerini ve sayılarını gizler.
plt.show()

### 3.2 Modeli tanımlamak

**Bu hücrede:** Katmanları sırayla dizerek modeli kuruyoruz: 28×28 görüntüyü düzleştiren bir katman, 128 nöronlu bir gizli katman ve 10 rakam için 10 nöronlu bir çıkış katmanı.

In [ ]:
model = keras.Sequential([                # Sequential → katmanları sırayla (üst üste) dizen model türü.
    keras.Input(shape=(28, 28)),          # Girdi: 28×28 piksellik bir görüntü.
    keras.layers.Flatten(),               # Flatten: 28×28 matrisi 784 sayılık düz bir vektöre çevirir.
    keras.layers.Dense(
        128,                              # 128 nöronlu gizli katman.
        activation="relu",                # Aktivasyon fonksiyonu: ReLU.
    ),                                    # Dense = tam bağlantılı: her nöron 784 girdinin hepsine bağlı.
    keras.layers.Dense(
        10,                               # Çıkış katmanı: 10 nöron, her rakam (0–9) için bir tane.
        activation="softmax",             # softmax: 10 çıktıyı toplamı 1 olan olasılıklara çevirir.
    ),
])

model.summary()   # Katmanları ve her katmandaki parametre (ağırlık + bias) sayısını tablo olarak gösterir.

**Sonuç:** Modelin **101.770** parametresi var. İlk `Dense` katmanında 784 × 128 ağırlık + 128 bias = 100.480 parametre bulunuyor. Eğitim, bu sayıların hepsini ayarlayacak.

### 3.3 Modeli derlemek ve eğitmek

- **`compile`:** Modelin nasıl öğreneceğini seçer: güncelleme yöntemi, kayıp fonksiyonu, başarı ölçüsü.
- **`fit`:** Eğitimi başlatır. Bölüm 2'deki döngü (ileri yayılım → kayıp → geri yayılım → güncelleme) burada otomatik çalışır.

**Bu hücrede:** Modeli derleyip 3 epok boyunca eğitiyoruz (Colab CPU'sunda yaklaşık 15 saniye).

In [ ]:
model.compile(
    optimizer="adam",                         # Ağırlık güncelleme yöntemi: gradyan inişinin gelişmiş hali.
    loss="sparse_categorical_crossentropy",   # Kayıp: çok sınıflı çapraz entropi.
                                              # "sparse" → etiketler 0–9 gibi tam sayılar olduğu için.
    metrics=["accuracy"],                     # Eğitim sırasında doğruluk oranını da göster.
)

gecmis = model.fit(
    X_egitim,              # Eğitim görüntüleri.
    y_egitim,              # Bu görüntülerin doğru rakamları.
    epochs=3,              # Epok: tüm eğitim verisinin baştan sona kaç kez dolaşılacağı.
    batch_size=128,        # Batch: her güncelleme adımında birlikte kullanılan görüntü sayısı.
    validation_split=0.1,  # Eğitim verisinin %10'unu ayırır; model onu görmeden başarısını izleriz.
)

**Sonuç:** Her epokta kayıp (`loss`) azalıyor, doğruluk (`accuracy`) artıyor. `val_accuracy`, modelin eğitimde görmediği görüntülerdeki başarısıdır. İki doğruluğun birbirine yakın olması modelin ezberlemediğini gösterir.

### 3.4 Modeli test etmek ve tahmin yaptırmak

**Bu hücrede:** Modeli hiç görmediği 10.000 test görüntüsüyle değerlendiriyor, ardından tek bir görüntü için tahmin yaptırıyoruz.

In [ ]:
kayip, dogruluk = model.evaluate(
    X_test,      # Modelin hiç görmediği test görüntüleri.
    y_test,      # Bu görüntülerin doğru rakamları.
    verbose=0,   # verbose=0 → ilerleme çubuğunu gösterme, sadece sonucu döndür.
)
print("Test doğruluğu: %", round(dogruluk * 100, 1))   # dogruluk 0–1 arası; ×100 → yüzde.

olasiliklar = model.predict(X_test[:1], verbose=0)   # predict → tahmin yapar.
# X_test[:1] → test setinin ilk görüntüsü. Sonuç: 10 rakamın her biri için bir olasılık.

tahmin = np.argmax(olasiliklar)          # np.argmax → en büyük olasılığın sırası = tahmin edilen rakam.
print("Modelin tahmini:", tahmin, "| Gerçek rakam:", y_test[0])
print("Model ne kadar emin: %", round(np.max(olasiliklar) * 100, 1))   # np.max → en büyük olasılık.

**Sonuç:** Yaklaşık 10 saniyelik eğitimle model test görüntülerinin yaklaşık **%97**'sini doğru tanıyor; ilk test görüntüsündeki 7 rakamını %99 eminlikle bildi. Aynı modeli PyTorch ile de kurabilirdik; sunumdaki karşılaştırma tablosu iki kütüphanenin aynı adımları nasıl yazdığını gösteriyor.

## 4. NLP'ye Giriş: Metni Sayılara Çevirmek

**Doğal Dil İşleme (NLP):** Bilgisayarların insan dilini anlaması ve üretmesi. Sinir ağları yalnızca sayılarla çalıştığı için metin iki adımda sayıya çevrilir:
1. **Tokenization:** Metni küçük parçalara (**token**) bölmek ve her parçaya bir numara (**ID**) vermek.
2. **Embedding:** Her token'ı, anlamını taşıyan bir sayı vektörüne çevirmek.

### 4.1 Tokenization

**Bu hücrede:** GPT-2 modelinin tokenizer'ını indirip bir İngilizce ve bir Türkçe cümleyi token'lara bölüyoruz.

In [ ]:
from transformers import AutoTokenizer   # transformers: Hugging Face'in hazır dil modeli kütüphanesi.
                                         # AutoTokenizer → modelin metni nasıl böldüğünü bilen araç.

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
# from_pretrained → adı verilen modelin tokenizer'ını internetten indirir.
# GPT-2: OpenAI'ın 2019'da yayımladığı, ağırlıklı olarak İngilizce metinle eğitilmiş model.

# Aşağıdaki işlemleri önce İngilizce, sonra Türkçe cümle için sırayla yap:
for cumle in ["The students went to the library.", "Öğrenciler kütüphaneye gittiler."]:
    idler = tokenizer.encode(cumle)               # encode → metni token ID'lerine (sayılara) çevirir.
    parcalar = [tokenizer.decode(i) for i in idler]
    # [... for i in idler] → her ID için aynı işlemi yapıp sonuçları yeni bir listeye toplar.
    # decode → ID'yi tekrar metne çevirir; böylece her token'ın hangi parça olduğunu görürüz.
    print(cumle)                                  # Cümlenin kendisini yazdır.
    print("  Token sayısı:", len(idler))          # len → listedeki eleman sayısı.
    print("  Parçalar    :", parcalar)
    print("  ID'ler      :", idler)

**Sonuç:** GPT-2 İngilizce cümleyi 7 token'a, neredeyse kelime kelime ayırdı. Aynı anlamdaki Türkçe cümleyi ise 17 parçaya böldü: model İngilizce ağırlıklı eğitildiği için Türkçe kelimeleri tanımıyor ve heceleyerek bölüyor. (`�` işareti, "Ö" gibi tek bir harfin iki parçaya bölündüğünü gösterir.) Token sayısı önemlidir: modelin okuyabildiği metin miktarı ve ticari LLM'lerin ücreti token ile ölçülür.

### 4.2 Embedding: anlamı sayılarla ifade etmek

Embedding, her token'ı veya cümleyi yüzlerce sayıdan oluşan bir **vektöre** çevirir. Bu vektörler eğitimle öğrenilir: **anlamca yakın metinlerin vektörleri birbirine yakın** olur. Yakınlığı **kosinüs benzerliği** ile ölçeriz: 1'e yakın = benzer, 0'a yakın = ilgisiz.

**Bu hücrede:** Türkçeyi de bilen hazır bir embedding modeliyle dört cümleyi vektöre çevirip birbirlerine ne kadar benzediklerini ölçüyoruz. Aynı modeli Bölüm 7'deki projede de kullanacağız.

In [ ]:
from sentence_transformers import SentenceTransformer   # Cümleleri vektöre çeviren hazır modeller.
from sentence_transformers import util                  # util → benzerlik hesabı gibi yardımcı araçlar.

model_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
# Türkçe dahil 50'den fazla dili bilen, her cümleyi 768 sayılık bir vektöre çeviren model (~1.1 GB).
# İçinde bir Transformer var (Bölüm 5): cümlenin anlamını tek bir vektörde özetler.

cumleler = [
    "Kedi koltukta uyuyor.",                  # 1. cümle
    "Kedicik kanepenin üstünde uyukluyor.",   # 2. cümle: 1. ile aynı anlam, farklı kelimeler
    "Borsa bugün sert düştü.",                # 3. cümle
    "Hisse senetleri büyük değer kaybetti.",  # 4. cümle: 3. ile aynı anlam, farklı kelimeler
]

vektorler = model_emb.encode(cumleler)          # encode → her cümleyi bir vektöre çevirir.
print("Vektörlerin boyutu:", vektorler.shape)   # (4, 768) → 4 cümle, her biri 768 sayı.

benzerlik = util.cos_sim(vektorler, vektorler)  # Her cümle çiftinin kosinüs benzerliğini hesaplar.
# Aynı listeyi iki kez veriyoruz: her cümleyi her cümleyle karşılaştırmak için (4 × 4 tablo).
# benzerlik[i][j] → i. cümle ile j. cümlenin benzerliği (sayma 0'dan başlar).
# float() → sonucu normal bir ondalık sayıya çevirir ki round ile yuvarlayabilelim.
print("Kedi ↔ kedi   :", round(float(benzerlik[0][1]), 2))
print("Borsa ↔ borsa :", round(float(benzerlik[2][3]), 2))
print("Kedi ↔ borsa  :", round(float(benzerlik[0][2]), 2))

**Sonuç:** Ortak kelimeleri olmamasına rağmen aynı konudaki cümleler belirgin bir benzerlik aldı (kedi–kedi 0.62, borsa–borsa 0.64); ilgisiz cümlelerin benzerliği ise 0'a yakın kaldı (−0.02). Embedding kelimelerin yazılışını değil, **anlamını** yakalar.

## 5. Transformer ve Attention

Bugünkü dil modellerinin tamamı **Transformer** mimarisine dayanır. Transformer'ın kalbi **attention** (dikkat) mekanizmasıdır: her kelime, anlamını belirlemek için cümledeki diğer kelimelere bakar.

### 5.1 Attention nasıl hesaplanır?

Her token üç vektör üretir:
- **Query (sorgu):** "Ben neyi arıyorum?"
- **Key (anahtar):** "Bende ne var?"
- **Value (değer):** "Bana bakana hangi bilgiyi aktarırım?"

Hesap üç adımdır: **(1)** sorgu ile anahtarları karşılaştırıp uyum puanı çıkar, **(2)** puanları softmax ile toplamı 1 olan ağırlıklara çevir, **(3)** değerlerin bu ağırlıklarla ortalamasını al.

**Bu hücrede:** "kedi süt içti" cümlesi için attention ağırlıklarını küçük sayılarla hesaplıyoruz. Gerçek modellerde Q, K, V vektörleri eğitimle öğrenilir; burada örnek olsun diye biz verdik.

In [ ]:
tokenlar = ["kedi", "süt", "içti"]   # Cümlenin 3 token'ı.

Q = np.array([[2.0, 0.0],     # Sorgu vektörleri (her satır bir token): "kedi" neyi arıyor?
              [0.0, 2.0],     # "süt" neyi arıyor?
              [2.0, 2.0]])    # "içti" neyi arıyor? → hem "kim?" hem "ne?" bilgisini arıyor.
K = np.array([[2.0, 0.0],     # Anahtar vektörleri: "kedi"de "kim?" bilgisi var.
              [0.0, 2.0],     # "süt"te "ne?" bilgisi var.
              [0.0, 0.0]])    # "içti"de ikisi de yok.

puanlar = Q @ K.T / np.sqrt(2)   # (1) Her token çifti için uyum puanı.
# K.T → K matrisinin devriği (satırlar sütun olur); Q @ K.T her sorguyu her anahtarla çarpar.
# np.sqrt(2) → vektör boyutunun (2) karekökü; puanlar aşırı büyümesin diye bölüyoruz.

agirliklar = np.exp(puanlar) / np.exp(puanlar).sum(axis=1, keepdims=True)   # (2) Softmax.
# np.exp → her puanı pozitif yapar. .sum(axis=1) → her SATIRI kendi içinde toplar.
# keepdims=True → bölme işleminin satır satır yapılabilmesi için boyutu korur.

print("Sütunlar: kedi, süt, içti")               # Sayıların hangi token'a ait olduğunu belirtir.
for i in range(3):                               # Her token için dikkat dağılımını yazdır.
    print(tokenlar[i], "→", np.round(agirliklar[i], 2))

**Sonuç:** "içti" dikkatini neredeyse eşit olarak "kedi" (kim içti?) ve "süt" (ne içildi?) token'larına verdi; kendisine neredeyse hiç bakmadı. Her satırın toplamı 1'dir. Adım (3)'te her token'ın yeni vektörü, değer (Value) vektörlerinin bu ağırlıklarla alınan ortalaması olur. Böylece "içti"nin yeni temsili "kedi" ve "süt" bilgisini de taşır. Gerçek bir modelde bu hesap her katmanda, onlarca **dikkat başlığıyla** (head) aynı anda yapılır.

### 5.2 Transformer mimarisi: GPT-2'nin içine bakış

Bir Transformer bloğu iki ana parçadan oluşur: **attention** (token'lar birbirinden bilgi toplar) ve **ileri beslemeli ağ** (her token'ın vektörü ayrı ayrı işlenir). Bu blok üst üste defalarca tekrarlanır.

**Bu hücrede:** GPT-2 modelini indirip yapısını gösteren ayarlarına bakıyoruz.

In [ ]:
from transformers import AutoModelForCausalLM   # Metin üreten (sonraki token'ı tahmin eden) modeli yükler.

gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")   # GPT-2'yi (~550 MB) indirir.
ayar = gpt2.config                              # config → modelin mimari ayarları.

print("Transformer blok sayısı :", ayar.n_layer)       # Üst üste konmuş blok sayısı.
print("Dikkat başlığı sayısı   :", ayar.n_head)        # Her blokta aynı anda çalışan attention sayısı.
print("Embedding boyutu        :", ayar.n_embd)        # Her token'ı temsil eden vektörün uzunluğu.
print("Sözlük boyutu           :", ayar.vocab_size)    # Modelin tanıdığı farklı token sayısı.
print("Bağlam penceresi        :", ayar.n_positions)   # Aynı anda okuyabildiği en fazla token (5.3).

parametre = gpt2.num_parameters()                      # Modeldeki toplam ağırlık + bias sayısı.
print("Toplam parametre        :", round(parametre / 1_000_000), "milyon")   # 1_000_000 = 1 milyon

**Sonuç:** GPT-2'nin küçük sürümünde 12 Transformer bloğu, her blokta 12 dikkat başlığı ve **124 milyon** parametre var. Her token 768 sayılık bir vektörle temsil ediliyor. Büyük modeller aynı yapıyı çok daha fazla blok ve parametreyle kurar (ör. GPT-3: 96 blok, 175 milyar parametre).

### 5.3 Context window (bağlam penceresi)

Bağlam penceresi, modelin **tek seferde okuyabildiği en fazla token sayısıdır**. Sohbet geçmişi, eklenen belgeler ve modelin cevabı bu pencereye sığmak zorundadır; sığmayan kısım model için yoktur. GPT-2'nin penceresi 1024 token, günümüzün büyük modellerinde ise yüz binlerce token'dır.

**Bu hücrede:** Uzun bir metnin kaç token olduğunu sayıp GPT-2'nin penceresine sığıp sığmadığını kontrol ediyoruz.

In [ ]:
metin = "Derin öğrenme ve büyük dil modelleri. " * 150   # Aynı cümleyi 150 kez yan yana ekler.

token_sayisi = len(tokenizer.encode(metin))   # Metni token'lara çevirip sayar (4.1'deki tokenizer).
pencere = gpt2.config.n_positions             # GPT-2'nin bağlam penceresi.

print("Metnin token sayısı :", token_sayisi)
print("Pencere boyutu      :", pencere)
print("Pencereye sığıyor mu?", token_sayisi <= pencere)   # <= → "küçük veya eşit mi?": True / False
print("Dışarıda kalan token:", max(0, token_sayisi - pencere))   # max(0, ...) → sonuç negatifse 0 yaz.

**Sonuç:** Metin 2701 token; GPT-2'nin 1024 token'lık penceresine sığmıyor ve 1677 token dışarıda kalıyor. Böyle bir metni modele vermek için ya kırpmak ya da parçalara bölmek gerekir. (Hücrenin çıktısında görebileceğiniz uyarı da metnin modelin sınırından uzun olduğunu hatırlatır.)

## 6. Büyük Dil Modelleri (LLM)

Bir LLM'in yaptığı iş şaşırtıcı derecede basittir: **metindeki bir sonraki token'ı tahmin etmek.** Tahmin edilen token metne eklenir ve işlem tekrarlanır. ChatGPT gibi sohbet asistanları da cevabı bu şekilde, token token üretir.

### 6.1 Sonraki token tahmini

**Bu hücrede:** GPT-2'ye yarım bir cümle veriyor ve sözlüğündeki 50.257 token'a verdiği olasılıklardan en yüksek 5 tanesine bakıyoruz.

In [ ]:
import torch   # PyTorch: Hugging Face modelleri PyTorch ile çalışır, çıktıları PyTorch tensörüdür.

girdi = tokenizer("Every morning I drink a cup of", return_tensors="pt")
# Metni token ID'lerine çevirir. return_tensors="pt" → sonucu PyTorch tensörü olarak ver.

with torch.no_grad():               # Eğitim yapmadığımız için türev hesabını kapatır (daha hızlı).
    cikti = gpt2(**girdi)           # Modeli çalıştırır. ** → girdi sözlüğündeki bilgileri modele aktarır.

puanlar = cikti.logits[0, -1]       # logits → her token için ham puan. [0, -1] → son konumun puanları.
olasiliklar = torch.softmax(puanlar, dim=0)   # softmax → puanları toplamı 1 olan olasılıklara çevirir.
                                              # dim=0 → işlemi vektörün tamamı üzerinde yap.
en_iyi = torch.topk(olasiliklar, 5)           # topk → en yüksek 5 olasılığı ve token ID'lerini verir.

for olasilik, token_id in zip(en_iyi.values, en_iyi.indices):
    # zip → iki listeyi eşleştirip birlikte dolaşır: (olasılık, ID) çiftleri.
    print(repr(tokenizer.decode(token_id)), "→ %", round(float(olasilik) * 100, 1))
    # repr → token'ı tırnak içinde yazar; baştaki boşluk da görünsün diye.

**Sonuç:** Model tek bir cevap vermedi, bir **olasılık dağılımı** üretti: `' coffee'` yaklaşık %47, `' tea'` %19... Bu dağılımı eğitildiği milyarlarca kelimelik metinden öğrendi. Token'ın başındaki boşluk da token'ın bir parçasıdır.

### 6.2 Metin üretimi: token token yazmak

**Bu hücrede:** Üretim döngüsünü elle yazıyoruz: her adımda en olası token'ı seçip metnin sonuna ekliyor ve modeli yeniden çalıştırıyoruz.

In [ ]:
metin = "Deep learning is"                           # Başlangıç metni.

for adim in range(8):                                # 8 yeni token üret.
    girdi = tokenizer(metin, return_tensors="pt")    # Güncel metni token'lara çevir.
    with torch.no_grad():                            # Türev hesabını kapat (6.1'deki gibi).
        puanlar = gpt2(**girdi).logits[0, -1]        # Bir sonraki token için puanlar.
    sonraki_id = torch.argmax(puanlar)               # argmax → en yüksek puanlı token'ın ID'si.
    metin = metin + tokenizer.decode(sonraki_id)     # Seçilen token'ı metnin sonuna ekle.
    print(adim + 1, ":", metin)                      # Metnin her adımda nasıl uzadığını göster.

**Sonuç:** Model metni token token uzattı. Sohbet uygulamalarında cevabın ekrana parça parça akmasının sebebi budur. Her zaman en olası token'ı seçmek (**greedy**) metni tekdüze yapar ve tekrara düşürebilir; bu yüzden gerçek uygulamalarda **sıcaklık (temperature)** ile olasılıklara göre seçim yapılır.

### 6.3 Sıcaklık (temperature) ile üretim

- **Düşük sıcaklık (ör. 0.3):** Model en olası token'lara yapışır → tutarlı ama tekdüze metin.
- **Yüksek sıcaklık (ör. 1.5):** Olasılıklar birbirine yaklaşır → çeşitli ama daha dağınık metin.

**Bu hücrede:** Hugging Face'in hazır `generate` fonksiyonuyla aynı başlangıçtan iki farklı sıcaklıkla metin üretiyoruz.

In [ ]:
girdi = tokenizer("Once upon a time,", return_tensors="pt")   # Hikâyenin başlangıcı.

for sicaklik in [0.3, 1.5]:                      # Önce düşük, sonra yüksek sıcaklık.
    torch.manual_seed(42)                        # Rastgeleliği sabitle: herkes aynı metni görsün.
    uretilen = gpt2.generate(
        **girdi,                                 # Başlangıç metninin token'ları.
        max_new_tokens=30,                       # En fazla 30 yeni token üret.
        do_sample=True,                          # En olasıyı değil, olasılıklara göre rastgele seç.
        temperature=sicaklik,                    # Sıcaklık: olasılık dağılımını keskinleştirir/düzleştirir.
        pad_token_id=tokenizer.eos_token_id,     # Teknik ayar: GPT-2'de "boşluk doldurma" token'ı yok,
    )                                            # metin sonu token'ını kullanmasını söylüyoruz.
    print("Sıcaklık", sicaklik, ":", tokenizer.decode(uretilen[0]))   # Üretilen ID'leri metne çevir.
    print()                                      # Boş satır: iki çıktıyı ayırmak için.

**Sonuç:** Düşük sıcaklıkta model hep en olası kelimeleri seçtiği için metin tekdüzeleşiyor ve kendini tekrar etmeye başlıyor. Yüksek sıcaklıkta metin daha çeşitli ama daha dağınık. Sohbet uygulamalarındaki "yaratıcılık" ayarı bu parametredir; genellikle ikisinin arasında bir değer (ör. 0.7) kullanılır.

### 6.4 GPT tipi modeller: temel model ve sohbet modeli

**GPT (Generative Pre-trained Transformer):** Çok büyük metinlerle, yalnızca sonraki token'ı tahmin etmek üzere **önceden eğitilmiş** Transformer.
- **Temel model (ör. GPT-2):** Talimat izlemeyi bilmez, metni sadece devam ettirir.
- **Sohbet modeli (ör. ChatGPT):** Aynı yapıdadır ama ön eğitimden sonra soru-cevap örnekleri ve insan geri bildirimiyle ayrıca eğitilmiştir.

**Bu hücrede:** Aynı soruyu önce temel model GPT-2'ye, sonra küçük bir sohbet modeline (Qwen2.5-0.5B-Instruct, ~1 GB) soruyoruz.

In [ ]:
soru = "What is the capital of Turkey?"   # İki modele de soracağımız soru: "Türkiye'nin başkenti neresi?"

# --- 1) Temel model: GPT-2 ---
girdi = tokenizer("Question: " + soru + "\nAnswer:", return_tensors="pt")   # Soru-cevap biçiminde ver.
uretilen = gpt2.generate(
    **girdi,                                # Sorunun token'ları.
    max_new_tokens=25,                      # En fazla 25 yeni token üret.
    do_sample=False,                        # Rastgelelik yok: her adımda en olası token.
    pad_token_id=tokenizer.eos_token_id,    # 6.3'teki teknik ayarın aynısı.
)
print("GPT-2 (temel model):")
print(tokenizer.decode(uretilen[0]))      # Üretilen token ID'lerini metne çevirip yazdır.
print()                                   # Boş satır.

# --- 2) Sohbet modeli: Qwen2.5-0.5B-Instruct ---
sohbet_adi = "Qwen/Qwen2.5-0.5B-Instruct"                        # Alibaba'nın küçük, açık sohbet modeli.
sohbet_tok = AutoTokenizer.from_pretrained(sohbet_adi)            # Bu modelin tokenizer'ı.
sohbet_model = AutoModelForCausalLM.from_pretrained(
    sohbet_adi,              # İndirilecek model.
    dtype=torch.float32,     # Sayı biçimi: 32 bitlik ondalık sayılar (CPU'da en uyumlu seçenek).
)

mesajlar = [{"role": "user", "content": soru}]    # Sohbet modelleri mesaj listesiyle çalışır: rol + içerik.
metin = sohbet_tok.apply_chat_template(
    mesajlar,                      # Gönderilecek mesajlar.
    tokenize=False,                # Şimdilik düz metin olarak ver (token'a sonra çevireceğiz).
    add_generation_prompt=True,    # Sonuna "şimdi asistan cevap verecek" işaretini ekle.
)
# apply_chat_template → mesajları, modelin eğitimde gördüğü özel sohbet biçimine çevirir.

girdi = sohbet_tok(metin, return_tensors="pt")   # Sohbet metnini token'lara çevir.
uretilen = sohbet_model.generate(
    **girdi,             # Sohbet metninin token'ları.
    max_new_tokens=40,   # En fazla 40 yeni token.
    do_sample=False,     # Her adımda en olası token.
)
soru_uzunlugu = len(girdi["input_ids"][0])   # Sohbet metnimiz kaç token? (input_ids → token ID listesi)
cevap = uretilen[0][soru_uzunlugu:]          # [soru_uzunlugu:] → o noktadan sonrası = modelin yeni yazdığı.
print("Qwen (sohbet modeli):")
print(sohbet_tok.decode(cevap, skip_special_tokens=True))   # skip_special_tokens → özel işaretleri gizle.

**Sonuç:** GPT-2 soruyu cevaplamak yerine metni devam ettirip kendini tekrar etti. Sohbet modeli ise kısa ve doğru bir cevap verdi. İki model de aynı işi yapıyor (sonraki token tahmini); fark, sohbet modelinin ön eğitimden sonra talimat izlemek üzere ayrıca eğitilmiş olmasıdır.

## 7. Proje: Müşteri Mesajlarını Otomatik Yönlendiren Yapay Zekâ

Şimdiye kadar öğrendiklerimizi **gerçek bir problemde** birleştiriyoruz. Bir şirketin müşteri hizmetlerine her gün binlerce mesaj gelir ve her mesajın doğru departmana gitmesi gerekir. Bu işi yapan bir yapay zekâ kuracağız:

1. **Embedding (Bölüm 4):** Her mesajı, içinde Transformer bulunan hazır bir modelle 768 sayılık bir anlam vektörüne çeviririz.
2. **Sinir ağı (Bölüm 1–3):** Bu vektörlerin üzerine Keras ile küçük bir sinir ağı eğitiriz; ağ, mesajın hangi departmana ait olduğunu öğrenir.
3. **Tahmin:** Yeni bir mesaj geldiğinde model departmanı ve ne kadar emin olduğunu söyler.

Hazır bir dil modelinin bilgisini alıp üzerine kendi küçük modelimizi eğitmeye **transfer öğrenme** denir. Gerçek projelerin çoğu bu şekilde kurulur: az veriyle, birkaç saniyede eğitilen ve işe yarayan bir model.

### 7.1 Eğitim verisi: örnek müşteri mesajları

**Bu hücrede:** 4 departman için 8'er örnek mesaj yazıyoruz (toplam 32 mesaj). Model bu örneklerden öğrenecek.

In [ ]:
ornekler = {
    "Fatura ve ödeme": [
        "Faturam neden bu kadar yüksek geldi?",
        "Kredi kartımdan iki kez ödeme alınmış.",
        "Ödeme yaparken hata alıyorum.",
        "Faturamı nereden görebilirim?",
        "Otomatik ödeme talimatı vermek istiyorum.",
        "Geçen ayın faturası hâlâ ödenmemiş görünüyor.",
        "Taksitli ödeme yapabilir miyim?",
        "Fatura adresimi değiştirmek istiyorum.",
    ],
    "Teknik destek": [
        "İnternetim sürekli kopuyor.",
        "Uygulama açılmıyor, hata veriyor.",
        "Şifremi unuttum, giriş yapamıyorum.",
        "Modemin ışıkları yanıp sönüyor.",
        "Mobil uygulama çok yavaş çalışıyor.",
        "Hesabıma giriş yaparken doğrulama kodu gelmiyor.",
        "Wi-Fi bağlantım çok zayıf.",
        "Güncellemeden sonra uygulama çöktü.",
    ],
    "Kargo ve teslimat": [
        "Siparişim hâlâ gelmedi.",
        "Kargom nerede?",
        "Teslimat adresini değiştirebilir miyim?",
        "Paket hasarlı geldi.",
        "Kargo takip numarası çalışmıyor.",
        "Siparişim yanlış adrese gitmiş.",
        "Teslimat ne zaman yapılacak?",
        "Kurye kapıya gelmeden gitmiş.",
    ],
    "İade ve iptal": [
        "Ürünü iade etmek istiyorum.",
        "Siparişimi iptal edebilir miyim?",
        "İade param ne zaman yatar?",
        "Aldığım ürün beden olarak olmadı.",
        "Aboneliğimi iptal etmek istiyorum.",
        "Ürün açıklamadaki gibi değil, geri göndereceğim.",
        "İade kargosu ücretli mi?",
        "Yanlış ürün geldi, değiştirmek istiyorum.",
    ],
}

departmanlar = list(ornekler)   # Sözlüğün anahtarlarından departman listesi; sıra numaraları 0, 1, 2, 3.

metinler = []                   # Tüm mesajları tek listede toplayacağız.
etiketler = []                  # Her mesajın departman numarası (modelin öğreneceği doğru cevap).
for numara, departman in enumerate(departmanlar):   # enumerate → hem sıra numarasını hem adı verir.
    for mesaj in ornekler[departman]:               # O departmanın her mesajı için:
        metinler.append(mesaj)                      #   mesajı listeye ekle,
        etiketler.append(numara)                    #   departman numarasını da etiket listesine ekle.

print("Toplam mesaj:", len(metinler))               # 4 departman × 8 mesaj = 32
print("Departmanlar:", departmanlar)

### 7.2 Mesajları anlam vektörlerine çevirmek

**Bu hücrede:** 32 mesajı, Bölüm 4.2'de yüklediğimiz embedding modeliyle vektöre çeviriyoruz. Sinir ağı metni değil, bu sayıları görecek.

In [ ]:
X = model_emb.encode(metinler)   # Her mesaj → 768 sayılık bir vektör (4.2'deki model).
y = np.array(etiketler)          # Etiket listesini NumPy dizisine çeviriyoruz; Keras bu biçimi ister.

print("X boyutu:", X.shape)      # (32, 768) → 32 mesaj, her biri 768 sayı.
print("y boyutu:", y.shape)      # (32,) → her mesaj için bir departman numarası.

### 7.3 Sinir ağını kurmak ve eğitmek

**Bu hücrede:** 768 sayıyı girdi olarak alan ve 4 departmandan birini seçen küçük bir sinir ağını Keras ile kurup eğitiyoruz. Bölüm 3'teki adımların aynısı: tanımla → derle → eğit.

In [ ]:
keras.utils.set_random_seed(42)       # Rastgele başlangıç ağırlıklarını sabitle: herkes aynı sonucu alsın.

siniflandirici = keras.Sequential([
    keras.Input(shape=(768,)),        # Girdi: bir mesajın 768 sayılık anlam vektörü.
    keras.layers.Dense(
        32,                           # 32 nöronlu gizli katman.
        activation="relu",            # Aktivasyon: ReLU.
    ),
    keras.layers.Dense(
        4,                            # Çıkış: 4 nöron, her departman için bir tane.
        activation="softmax",         # softmax → 4 departmana toplamı 1 olan olasılıklar verir.
    ),
])

siniflandirici.compile(
    optimizer="adam",                          # Ağırlık güncelleme yöntemi.
    loss="sparse_categorical_crossentropy",    # Çok sınıflı çapraz entropi (etiketler 0–3 tam sayı).
    metrics=["accuracy"],                      # Doğruluk oranını da takip et.
)

gecmis = siniflandirici.fit(
    X,              # Girdi: mesaj vektörleri.
    y,              # Doğru cevaplar: departman numaraları.
    epochs=60,      # 32 örneği 60 kez dolaş (veri çok küçük olduğu için birkaç saniye sürer).
    verbose=0,      # Her epoğu ekrana yazma; yalnızca sonuca bakacağız.
)

son_dogruluk = gecmis.history["accuracy"][-1]    # history → her epoğun sonuçları; [-1] → son epok.
print("Eğitim doğruluğu: %", round(son_dogruluk * 100))

### 7.4 Yeni mesajları yönlendirmek

**Bu hücrede:** Modele eğitimde **hiç görmediği** mesajları veriyor, hangi departmana gönderdiğine ve ne kadar emin olduğuna bakıyoruz.

In [ ]:
def yonlendir(mesaj):                           # Bir mesajı alıp departmanını tahmin eden fonksiyon.
    vektor = model_emb.encode([mesaj])          # 1) Mesajı anlam vektörüne çevir (liste içinde tek mesaj).
    olasiliklar = siniflandirici.predict(vektor, verbose=0)[0]   # 2) 4 departman için olasılıklar.
    en_iyi = np.argmax(olasiliklar)             # 3) En yüksek olasılıklı departmanın sırası.
    guven = round(float(olasiliklar[en_iyi]) * 100)             # Güven: olasılığın yüzde karşılığı.
    return departmanlar[en_iyi], guven          # Departman adını ve güveni döndür.

yeni_mesajlar = [                               # Eğitim verisinde OLMAYAN mesajlar.
    "Bu ay faturama fazladan ücret yansımış.",
    "Telefonumdaki uygulama sürekli kapanıyor.",
    "Paketim üç gündür yolda, ne zaman gelir?",
    "Siparişimden vazgeçtim.",
    "Hesabım kilitlendi.",
]

for mesaj in yeni_mesajlar:                     # Her yeni mesaj için:
    departman, guven = yonlendir(mesaj)         #   departmanı ve güveni tahmin et,
    print(mesaj, "→", departman, "| güven: %", guven)   #   sonucu yazdır.

**Sonuç:** Model, eğitimde hiç görmediği mesajları doğru departmanlara gönderdi. Örneğin "Hesabım kilitlendi" cümlesinde "şifre" ya da "giriş" kelimesi geçmiyor ama model, anlamından bunun teknik destek konusu olduğunu çıkardı. Bunu mümkün kılan, embedding modelinin kelimelerin yazılışını değil **anlamını** yakalamasıdır. Güveni düşük çıkan mesajlar gerçek bir sistemde bir insana yönlendirilebilir.

### 7.5 Kendi mesajınızı deneyin

**Bu hücrede:** Sağ tarafta çıkan metin kutusuna istediğiniz müşteri mesajını yazın ve hücreyi yeniden çalıştırın (`Shift + Enter`).

In [ ]:
mesaj = "Kargo görevlisi paketi komşuya bırakmış."  # @param {type:"string"}
# "# @param" → Colab bu satırı sağ tarafta düzenlenebilir bir metin kutusu olarak gösterir.

departman, guven = yonlendir(mesaj)             # 7.4'teki fonksiyonla tahmin et.
print("Mesaj     :", mesaj)
print("Departman :", departman)
print("Güven     : %", guven)

**Sonuç:** 32 örnek mesaj ve birkaç saniyelik eğitimle, Türkçe müşteri mesajlarını anlayıp yönlendiren çalışan bir yapay zekâ kurduk. Daha fazla örnek ve departman eklendikçe model güçlenir. Aynı yöntemle e-posta sınıflandırma, şikâyet tespiti ve duygu analizi gibi birçok gerçek uygulama kurulabilir.

## Özet

| Konu | Tek cümlede |
|---|---|
| Deep learning | Çok katmanlı sinir ağlarıyla kuralları veriden öğrenmek. |
| Nöron | Girdileri ağırlıklarla çarpıp toplar, bias ekler, aktivasyondan geçirir. |
| Katman ve ağ | Nöronlar katmanlar halinde dizilir; bir katman = bir matris çarpımı. |
| Aktivasyon | Ağa doğrusal olmayan davranış kazandırır (ReLU, sigmoid, tanh). |
| İleri yayılım | Girdiden tahmine giden hesap. |
| Kayıp | Tahminin ne kadar yanlış olduğunu ölçer (MSE, çapraz entropi). |
| Geri yayılım | Her ağırlığın hataya etkisini (gradyan) bulur; gradyan inişi ağırlıkları günceller. |
| Keras / PyTorch | Bu hesapların hepsini otomatik yapan kütüphaneler. |
| Tokenization | Metni token'lara ve numaralara çevirir. |
| Embedding | Token'ları anlam taşıyan vektörlere çevirir. |
| Attention | Her token'ın diğer token'lardan ne kadar bilgi alacağını belirler. |
| Transformer | Attention ve ileri beslemeli ağ bloklarının üst üste dizilmesi. |
| Context window | Modelin tek seferde okuyabildiği en fazla token sayısı. |
| LLM | Sonraki token'ı tahmin ederek metni token token üretir. |
| GPT tipi modeller | Önceden eğitilmiş, metin üreten Transformer; sohbet modelleri ayrıca talimatla eğitilir. |
| Proje | Hazır bir dil modelinin embedding'leri + Keras ile eğitilen küçük bir ağ = çalışan bir mesaj yönlendirici. |